In [1]:
!pip install kagglehub

In [2]:
import kagglehub
import os

print("Downloading datasets... this might take a few minutes.")

# 1. Download Coursera Dataset
coursera_path = kagglehub.dataset_download("septa97/100k-courseras-course-reviews-dataset")
print(f"Coursera data saved to: {coursera_path}")

# 2. Download ArXiv Dataset (Warning: This is a large dataset)
arxiv_path = kagglehub.dataset_download("Cornell-University/arxiv")
print(f"ArXiv data saved to: {arxiv_path}")

# 3. Download Wikipedia SQLite Dataset
wiki_path = kagglehub.dataset_download("jkkphys/english-wikipedia-articles-20170820-sqlite")
print(f"Wikipedia data saved to: {wiki_path}")

100%|██████████| 12.2M/12.2M [00:00<00:00, 45.4MB/s]

Extracting files...


Coursera data saved to: /root/.cache/kagglehub/datasets/septa97/100k-courseras-course-reviews-dataset/versions/2
Using Colab cache for faster access to the 'arxiv' dataset.
ArXiv data saved to: /kaggle/input/arxiv


100%|██████████| 6.65G/6.65G [01:16<00:00, 93.1MB/s]

Extracting files...


Wikipedia data saved to: /root/.cache/kagglehub/datasets/jkkphys/english-wikipedia-articles-20170820-sqlite/versions/3


In [3]:
!wget https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz
!gunzip conceptnet-assertions-5.7.0.csv.gz

--2026-02-25 13:50:55--  https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.48.118, 52.217.234.168, 16.182.96.32, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.48.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 497963447 (475M) [application/x-gzip]
Saving to: ‘conceptnet-assertions-5.7.0.csv.gz’

conceptnet-assertio 100%[===================>] 474.89M  45.7MB/s    in 11s     

2026-02-25 13:51:06 (43.9 MB/s) - ‘conceptnet-assertions-5.7.0.csv.gz’ saved [497963447/497963447]



In [4]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import sqlite3
import spacy
import json
import pandas as pd
import sqlite3 as sqlite_external

# 1. Load your NLP model and connect to your custom DB
nlp = spacy.load("en_core_web_sm")
db_conn = sqlite3.connect("knowledge_graph.db")

# 2. Re-declare your processing function from earlier (simplified for speed)
def process_text_chunk(text_list, conn):
    cursor = conn.cursor()
    for text in text_list:
        if not text: continue
        doc = nlp(text.lower()[:5000]) # Cap length to avoid freezing on massive paragraphs

        concepts = []
        for chunk in doc.noun_chunks:
            if chunk.root.pos_ not in ['PRON', 'DET']:
                concept = " ".join([t.lemma_ for t in chunk if not t.is_stop and t.is_alpha])
                if len(concept) > 2: concepts.append(concept)

        concepts = list(set(concepts))

        # Insert Concepts
        for concept in concepts:
            cursor.execute("INSERT OR IGNORE INTO concepts (name) VALUES (?)", (concept,))
            cursor.execute("UPDATE concepts SET frequency = frequency + 1 WHERE name = ?", (concept,))

    conn.commit()

# 3. Tool for chunking ArXiv JSON data
def process_arxiv_chunks(json_path, db_conn, chunk_size=500, max_chunks=10):
    print("Starting ArXiv processing...")
    texts = []
    chunk_count = 0

    with open(json_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            # Grab the abstract/summary of the paper
            texts.append(data.get('abstract', ''))

            if len(texts) >= chunk_size:
                print(f"Processing ArXiv chunk {chunk_count + 1}...")
                process_text_chunk(texts, db_conn)
                texts = [] # Empty the RAM
                chunk_count += 1
                if chunk_count >= max_chunks:
                    print("Reached max chunks for ArXiv test.")
                    break

# 4. Tool for chunking Coursera CSV data
def process_coursera_chunks(csv_path, db_conn, chunk_size=500, max_chunks=10):
    print("\nStarting Coursera processing...")
    chunk_count = 0
    # Read the CSV in chunks
    for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
        # Assuming the review text is in a column named 'Review' or 'text'
        # We will just grab the first string column we find for safety
        text_col = chunk.select_dtypes(include=['object']).columns[0]
        texts = chunk[text_col].dropna().tolist()

        print(f"Processing Coursera chunk {chunk_count + 1}...")
        process_text_chunk(texts, db_conn)

        chunk_count += 1
        if chunk_count >= max_chunks:
            print("Reached max chunks for Coursera test.")
            break

print("Tools loaded! Ready for Cell 2.")

Tools loaded! Ready for Cell 2.


In [3]:
import kagglehub
import os

# 1. Get the exact folder path for Coursera
coursera_folder = kagglehub.dataset_download("septa97/100k-courseras-course-reviews-dataset")
print("--- COURSERA ---")
print(f"YOUR FOLDER PATH IS: {coursera_folder}")
print("Files inside this folder:")
print(os.listdir(coursera_folder))

# 2. Get the exact folder path for ArXiv
arxiv_folder = kagglehub.dataset_download("Cornell-University/arxiv")
print("\n--- ARXIV ---")
print(f"YOUR FOLDER PATH IS: {arxiv_folder}")
print("Files inside this folder:")
print(os.listdir(arxiv_folder))

Using Colab cache for faster access to the '100k-courseras-course-reviews-dataset' dataset.
--- COURSERA ---
YOUR FOLDER PATH IS: /kaggle/input/100k-courseras-course-reviews-dataset
Files inside this folder:
['reviews.csv', 'reviews_by_course.csv']
Using Colab cache for faster access to the 'arxiv' dataset.

--- ARXIV ---
YOUR FOLDER PATH IS: /kaggle/input/arxiv
Files inside this folder:
['arxiv-metadata-oai-snapshot.json']


In [7]:
import itertools

def process_text_chunk(text_list, conn):
    cursor = conn.cursor()
    for text in text_list:
        if not text: continue
        # Cap length to avoid freezing on massive paragraphs
        doc = nlp(text.lower()[:5000])

        concepts = []
        for chunk in doc.noun_chunks:
            if chunk.root.pos_ not in ['PRON', 'DET']:
                concept = " ".join([t.lemma_ for t in chunk if not t.is_stop and t.is_alpha])
                if len(concept) > 2: concepts.append(concept)

        concepts = list(set(concepts))
        concept_ids = []

        # 1. Insert Concepts & Grab their IDs
        for concept in concepts:
            cursor.execute("INSERT OR IGNORE INTO concepts (name) VALUES (?)", (concept,))
            cursor.execute("UPDATE concepts SET frequency = frequency + 1 WHERE name = ?", (concept,))

            # Fetch the ID of the concept we just inserted/updated
            cursor.execute("SELECT id FROM concepts WHERE name = ?", (concept,))
            row = cursor.fetchone()
            if row:
                concept_ids.append(row[0])

        # 2. Build Relationships (Edges)
        if len(concept_ids) > 1:
            # We cap this at 50 concepts per chunk so the math doesn't explode your RAM
            if len(concept_ids) < 50:
                pairs = list(itertools.combinations(sorted(concept_ids), 2))
                for source, target in pairs:
                    cursor.execute('''
                        INSERT INTO relationships (source_id, target_id, relation_type)
                        VALUES (?, ?, 'co_occurrence')
                        ON CONFLICT(source_id, target_id) DO UPDATE SET weight = weight + 1
                    ''', (source, target))

    conn.commit()

print("Relationship builder loaded! Run your Kaggle chunker cell (Cell 2) one more time.")

Relationship builder loaded! Run your Kaggle chunker cell (Cell 2) one more time.


In [5]:
import os

# Grab the cursor
cursor = db_conn.cursor()

# Build the 'concepts' table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS concepts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        frequency INTEGER DEFAULT 1
    )
''')

# Build the 'relationships' table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS relationships (
        source_id INTEGER,
        target_id INTEGER,
        weight INTEGER DEFAULT 1,
        relation_type TEXT,
        PRIMARY KEY (source_id, target_id),
        FOREIGN KEY (source_id) REFERENCES concepts (id),
        FOREIGN KEY (target_id) REFERENCES concepts (id)
    )
''')

# Save the changes
db_conn.commit()
print("BOOM. Tables created successfully. The database is ready for data.")

BOOM. Tables created successfully. The database is ready for data.


In [8]:
import os

# --- PASTE YOUR EXACT KAGGLEHUB PATHS HERE ---
# Look at the output from your kagglehub download cell and paste those folder paths here.
# It usually looks something like '/root/.cache/kagglehub/datasets/...'

COURSERA_FOLDER = "/kaggle/input/100k-courseras-course-reviews-dataset"
ARXIV_FOLDER = "/kaggle/input/arxiv"

# Find the exact files inside those folders
coursera_file = os.path.join(COURSERA_FOLDER, "reviews.csv") # Check exact filename in the folder
arxiv_file = os.path.join(ARXIV_FOLDER, "arxiv-metadata-oai-snapshot.json")

# RUN THE CHUNKER!
# We are limiting it to 5 chunks of 500 texts each just to test it without you waiting 3 hours.
try:
    process_coursera_chunks(coursera_file, db_conn, chunk_size=500, max_chunks=5)
    process_arxiv_chunks(arxiv_file, db_conn, chunk_size=500, max_chunks=5)
    print("FINISHED! Database updated successfully.")
except Exception as e:
    print(f"An error occurred. Check your file paths! Error: {e}")

# Check how many concepts you captured!
cursor = db_conn.cursor()
cursor.execute("SELECT COUNT(*) FROM concepts")
count = cursor.fetchone()[0]
print(f"\nBOOM. You now have {count} unique concepts sitting in your SQLite database.")


Starting Coursera processing...
Processing Coursera chunk 1...
Processing Coursera chunk 2...
Processing Coursera chunk 3...
Processing Coursera chunk 4...
Processing Coursera chunk 5...
Reached max chunks for Coursera test.
Starting ArXiv processing...
Processing ArXiv chunk 1...
Processing ArXiv chunk 2...
Processing ArXiv chunk 3...
Processing ArXiv chunk 4...
Processing ArXiv chunk 5...
Reached max chunks for ArXiv test.
FINISHED! Database updated successfully.

BOOM. You now have 38972 unique concepts sitting in your SQLite database.


In [14]:
import sqlite3
from IPython.display import FileLink, display

# 1. Reconnect to the database (unlocking the door)
db_conn = sqlite3.connect("knowledge_graph.db")
cursor = db_conn.cursor()

# 2. Check the Relationships (Edges)
cursor.execute("SELECT COUNT(*) FROM relationships")
edge_count = cursor.fetchone()[0]
print(f"SUCCESS: Your graph has {edge_count} relationships connecting your concepts!")

# 3. Close it safely again
db_conn.close()



SUCCESS: Your graph has 756021 relationships connecting your concepts!
